In [1]:
import os
import sys

sys.path.append("../../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.load_model import load_model
from utils.model_utils.save_module import save_module
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune import (
    prune_concern_identification,
)

In [3]:
name = "bert-6-128-yahoo"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.3
seed = 44
include_layers = ["attention"]
exclude_layers = None

In [4]:
script_start_time = datetime.now()
print(f"Script started at: {script_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

Script started at: 2024-09-13 03:02:36


In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.

{'model_name': 'Models/bert-6-128-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}

The model Models/bert-6-128-yahoo is loaded.

In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    do_cache=True,
    seed=seed,
)

Loading cached dataset YahooAnswersTopics.

train.pkl is loaded from cache.

valid.pkl is loaded from cache.

test.pkl is loaded from cache.

The dataset YahooAnswersTopics is loaded

{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}

In [7]:
# print("Evaluate the original model")
# result = evaluate_model(model, model_config, test_dataloader)

In [8]:
for concern in range(num_labels):
    train = copy.deepcopy(train_dataloader)
    valid = copy.deepcopy(valid_dataloader)
    positive_samples = SamplingDataset(
        train,
        concern,
        num_samples,
        num_labels,
        True,
        4,
        device=device,
        resample=False,
        seed=seed,
    )
    negative_samples = SamplingDataset(
        train,
        concern,
        num_samples,
        num_labels,
        False,
        4,
        device=device,
        resample=False,
        seed=seed,
    )
    all_samples = SamplingDataset(
        train,
        200,
        num_samples,
        num_labels,
        False,
        4,
        device=device,
        resample=False,
        seed=seed,
    )

    module = copy.deepcopy(model)

    prune_concern_identification(
        module,
        model_config,
        positive_samples,
        negative_samples,
        include_layers=include_layers,
        exclude_layers=exclude_layers,
        sparsity_ratio=ci_ratio,
    )

    print(f"Evaluate the pruned model {concern}")
    result = evaluate_model(module, model_config, test_dataloader)
    get_sparsity(module)

    similar(
        model, module, valid, concern, num_samples, num_labels, device=device, seed=seed
    )

    # save_module(module, "Modules/", f"ci_{name}_{ci_ratio}p.pt")

Evaluate the pruned model 0

Evaluating the model:   0%|                                          | 0/1875 [00:00<?, ?it/s]

Loss: 1.2142

Precision: 0.6487, Recall: 0.6156, F1-Score: 0.6207

              precision    recall  f1-score   support

           0       0.55      0.47      0.50      2941
           1       0.70      0.49      0.58      2997
           2       0.69      0.65      0.67      3016
           3       0.33      0.65      0.44      2978
           4       0.72      0.76      0.74      3017
           5       0.84      0.77      0.80      3004
           6       0.66      0.40      0.50      3037
           7       0.64      0.61      0.63      3026
           8       0.60      0.71      0.65      2997
           9       0.75      0.65      0.69      2987

    accuracy                           0.62     30000
   macro avg       0.65      0.62      0.62     30000
weighted avg       0.65      0.62      0.62     30000


...

CCA coefficients mean non-concern: (0.999567844940433, 0.999567844940433)

Linear CKA concern: 0.9997553288626215

Linear CKA non-concern: 0.99965271312886

Kernel CKA concern: 0.9992002514386131

Kernel CKA non-concern: 0.9989465029481788